In [ ]:
# Topic: CovariateExamples
# Execution group: smoke
# Workflow family: signal
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/CovariateExamples.md


In [ ]:
import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "CovariateExamples"
FAMILY = "signal"
np.random.seed(2026)
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"CovariateExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"CovariateExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"CovariateExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"CovariateExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "t=0:.01:5; t=t';",
    "x=exp(-t);",
    "y=sin(2*pi*t);",
    "z=(-y).^3;",
    "fx=abs(y);",
    "fy=abs(y).^2;",
    "dLabels1={'f_x','f_y'};",
    "dLabels2={'x','y','z'};",
    "plotProps = {{' ''g'', ''LineWidth'' ,.5'},... %for x",
    "{' ''k'', ''LineWidth'' ,.5'},...   %for y",
    "{' ''b'' '}}; %for z",
    "force = Covariate(t, [fx fy], 'Force', 'time', 's', 'N', dLabels1);",
    "position=Covariate(t,[x y z], 'Position','time','s','cm', dLabels2);",
    "position.getSigRep.plot('all',plotProps); %same as position.plot",
    "plotPropsForce = {{' ''b'' '},{' ''k'' '}};",
    "figure;",
    "subplot(1,2,1); force.getSigRep.plot('all',plotPropsForce);",
    "subplot(1,2,2); force.getSigRep('zero-mean').plot('all',plotPropsForce);"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for CovariateExamples.")


In [ ]:
# CovariateExamples: build and inspect multiple covariate signals.
t = np.arange(0.0, 5.0 + 0.01, 0.01); x = np.exp(-t); y = np.sin(2.0 * np.pi * t); z = (-y) ** 3
force = Covariate(time=t, data=np.column_stack([np.abs(y), np.abs(y) ** 2]), name="Force", labels=["f_x", "f_y"])
position = Covariate(time=t, data=np.column_stack([x, y, z]), name="Position", labels=["x", "y", "z"])
force_zero_mean = force.data - np.mean(force.data, axis=0, keepdims=True)

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
axes[0, 0].plot(t, position.data[:, 0], "g", linewidth=0.6, label="x")
axes[0, 0].plot(t, position.data[:, 1], "k", linewidth=0.6, label="y")
axes[0, 0].plot(t, position.data[:, 2], "b", linewidth=0.6, label="z")
axes[0, 0].set_title(f"{TOPIC}: position covariates"); axes[0, 0].legend(loc="upper right")
axes[0, 1].plot(t, force.data[:, 0], "b", linewidth=1.0, label="f_x")
axes[0, 1].plot(t, force.data[:, 1], "k", linewidth=1.0, label="f_y")
axes[0, 1].set_title("Force (original)"); axes[0, 1].legend(loc="upper right")
axes[1, 0].plot(t, force_zero_mean[:, 0], "b", linewidth=1.0, label="f_x")
axes[1, 0].plot(t, force_zero_mean[:, 1], "k", linewidth=1.0, label="f_y")
axes[1, 0].set_title("Force (zero-mean)"); axes[1, 0].legend(loc="upper right")
axes[1, 1].plot(t, position.data[:, 1], "k", linewidth=1.0); axes[1, 1].set_title("Position y")
for ax in axes.ravel(): ax.set_xlabel("time [s]")
plt.tight_layout(); plt.show()

assert position.data.shape == (t.size, 3)
assert force.data.shape == (t.size, 2)
assert np.isfinite(force_zero_mean).all()
CHECKPOINT_METRICS = {"position_var": float(np.var(position.data[:, 1])), "force_mean": float(np.mean(force.data[:, 0]))}
CHECKPOINT_LIMITS = {"position_var": (0.05, 2.0), "force_mean": (0.0, 2.0)}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
